# STEP 0: 라이브러리 import 및 날짜 지정

In [1129]:
import pandas as pd
import numpy as np
import warnings
import folium
import folium as fo

import psycopg2
import datetime
from datetime import timedelta, timezone, datetime, date
from dateutil.relativedelta import relativedelta
import time

import requests
import json
from pandas.io.json import json_normalize

import pytz
from zeep import Client

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

#---API 적용 시에 필수항목은 아님
from multiprocessing.pool import Pool

#--- GFS & LDAPS자료를 불러오는데 필요한 모듈
from wolke.loader.nwp import GFSAPI, GFSLoader, LDAPSLoader, GFSDayAhead
# from nwp_hshan import GFSAPI, GFSLoader, LDAPSLoader, GFSDayAhead
from wolke.loader.interpolate import Interpolator, GHIInterpolator

#--- 일사량 보정 알고리즘 (실질적으로 수정된 내용이 여기 포함되어 있음)
from wolke.loader.postprocess import GnLCSICorrector

#--- 일사량을 발전량으로 변환시키는데 필요한 모듈 
from wolke.loader.pvstandard import PreprocessData
from wolke.pvsim.pvskeleton import PVSkeleton

import warnings
warnings.filterwarnings('ignore')

In [1130]:
tz_KST = timezone(timedelta(hours=9))
chasu = 1 # 1차 예측 ? 2차 예측 ?

start_year = 2023
start_month = 6
start_day = 22
target_start_dt = datetime(start_year, start_month, start_day, 0, 0, 0, 0, tzinfo=tz_KST)

end_year = 2023
end_month = 6
end_day = 22
target_end_dt = datetime(end_year, end_month, end_day, 0, 0, 0, 0, tzinfo=tz_KST)

print(target_start_dt, target_end_dt, sep='\n')

2023-06-22 00:00:00+09:00
2023-06-22 00:00:00+09:00


# STEP 1: HAEZOOM Day-Ahead

In [1131]:
# token 정보
haezoom_token = 'aGFlem9vbTppZG5zdG9yeTE4Mw==' #token 없는 경우 존재할 수도 있음 -> 거의 Lasee, haezoom, test_mj 중에 하나일텐데
testmj_token = 'dGVzdF9tajptajEyMzQ1Njc4OQ=='

# -- 원하는 개별자원 id 지정 (여러개도 가능)
target_id_list = [10602]

In [1132]:
def predict_datetime(chasu, target_start_dt, target_end_dt, id_list):
    start = time.time()

    url = 'https://api.haezoom.io/v1/test/predictDatetime/'
    headers = {
            "Authorization": "Basic "+haezoom_token,
        'Content-Type': 'application/json; charset=utf-8'
        }
    
    start_dt = target_start_dt
    res_list = []

    while start_dt <= target_end_dt:
        start2 = time.time()

        for target_id in target_id_list:
            end_dt = start_dt + timedelta(days=1)
            ref_dt = start_dt - timedelta(days=1) + timedelta(hours=9) if chasu==1 else start_dt - timedelta(days=1) + timedelta(hours=16) 

            params = {'pid': target_id, 'start':start_dt, 'end': end_dt, 'ref': ref_dt}
            res = requests.get(url, headers=headers, params = params)
            res_list.append(res)
        print('[{}]\nstart: {}\nend: {}\nref: {}'.format(res,start_dt, end_dt, ref_dt))
        start_dt = start_dt + timedelta(days=1)
        end2 = time.time()
        print(f"{end2 - start2:.5f} sec")

    end = time.time()
    print(f"total : {end - start:.5f} sec")
        
    return res_list

In [1133]:
def make_analysis_format(res_list, chasu):

    res_json_list = pd.DataFrame([each_res.json() for each_res in res_list])
    form_df = pd.DataFrame(columns=['tilt', 'azimuth', 'date', 'plant_id', 'plant_name', 'capacity', 'predict_yield', 'ref_h'])
    display(res_json_list)
    display(form_df)

    for i, _res in res_json_list.iterrows():
        res = _res['result']

        tilt = res['conditions']['applied']['tilt'][0]
        azi = res['conditions']['applied']['azimuth'][0]
        date = res['predict']['date']
        plant_id = res['predict']['plant']['id']
        plant_name = res['predict']['plant']['name']
        predict_yield = res['predict']['plant']['predictYield']['data']
        ref_h = datetime.strptime(res['conditions']['ref'], '%Y-%m-%dT%H:%M:%S+00:00').hour

        total_capacity = 0

        for tmp_inv in res['predict']['inverters']:
            total_capacity += tmp_inv['capacity']
        form_df.loc[i] = [tilt, azi, date, plant_id, plant_name,total_capacity, predict_yield, ref_h]
        
    form_df['chasu'] = chasu
    return form_df

In [1134]:
chasu = 1

res_list_1 = predict_datetime(chasu=chasu, target_start_dt=target_start_dt, target_end_dt=target_end_dt, id_list=target_id_list)
df_1 = make_analysis_format(res_list_1, chasu)
df_1.head()

[<Response [200]>]
start: 2023-06-22 00:00:00+09:00
end: 2023-06-23 00:00:00+09:00
ref: 2023-06-21 09:00:00+09:00
0.63505 sec
total : 0.63509 sec


,code,result,status
0,200,"{'conditions': {'id': '10602', 'timestampType'...",success


,tilt,azimuth,date,plant_id,plant_name,capacity,predict_yield,ref_h


,tilt,azimuth,date,plant_id,plant_name,capacity,predict_yield,ref_h,chasu
0,15.0,180.0,"[2023-06-22T01:00:00+09:00, 2023-06-22T02:00:0...",10602,신안 임자태양광발전소,99986.25,"[0.0, 0.0, 0.0, 0.0, 0.0, 303.3125, 4484.0, 15...",12,1


In [1135]:
chasu = 2

res_list_2 = predict_datetime(chasu=chasu, target_start_dt=target_start_dt, target_end_dt=target_end_dt, id_list=target_id_list)
df_2 = make_analysis_format(res_list_2, chasu)
df_2.head()

[<Response [200]>]
start: 2023-06-22 00:00:00+09:00
end: 2023-06-23 00:00:00+09:00
ref: 2023-06-21 16:00:00+09:00
0.57157 sec
total : 0.57161 sec


,code,result,status
0,200,"{'conditions': {'id': '10602', 'timestampType'...",success


,tilt,azimuth,date,plant_id,plant_name,capacity,predict_yield,ref_h


,tilt,azimuth,date,plant_id,plant_name,capacity,predict_yield,ref_h,chasu
0,15.0,180.0,"[2023-06-22T01:00:00+09:00, 2023-06-22T02:00:0...",10602,신안 임자태양광발전소,99986.25,"[0.0, 0.0, 0.0, 0.0, 0.0, 279.75, 4489.0, 1413...",0,2


In [1136]:
display(df_1.head())
display(df_1.tail())

,tilt,azimuth,date,plant_id,plant_name,capacity,predict_yield,ref_h,chasu
0,15.0,180.0,"[2023-06-22T01:00:00+09:00, 2023-06-22T02:00:0...",10602,신안 임자태양광발전소,99986.25,"[0.0, 0.0, 0.0, 0.0, 0.0, 303.3125, 4484.0, 15...",12,1


,tilt,azimuth,date,plant_id,plant_name,capacity,predict_yield,ref_h,chasu
0,15.0,180.0,"[2023-06-22T01:00:00+09:00, 2023-06-22T02:00:0...",10602,신안 임자태양광발전소,99986.25,"[0.0, 0.0, 0.0, 0.0, 0.0, 303.3125, 4484.0, 15...",12,1


In [1137]:
display(df_2.head())
display(df_2.tail())

,tilt,azimuth,date,plant_id,plant_name,capacity,predict_yield,ref_h,chasu
0,15.0,180.0,"[2023-06-22T01:00:00+09:00, 2023-06-22T02:00:0...",10602,신안 임자태양광발전소,99986.25,"[0.0, 0.0, 0.0, 0.0, 0.0, 279.75, 4489.0, 1413...",0,2


,tilt,azimuth,date,plant_id,plant_name,capacity,predict_yield,ref_h,chasu
0,15.0,180.0,"[2023-06-22T01:00:00+09:00, 2023-06-22T02:00:0...",10602,신안 임자태양광발전소,99986.25,"[0.0, 0.0, 0.0, 0.0, 0.0, 279.75, 4489.0, 1413...",0,2


In [1138]:
df_1.rename(columns = {'predict_yield':'pred_1'}, inplace=True)
df_2.rename(columns = {'predict_yield':'pred_2'}, inplace=True)

In [1139]:
df = df_1.copy()
df.drop(columns=['chasu'], inplace=True)
df['pred_2'] = df_2['pred_2']

In [1140]:
df_stack=[df[x].apply(pd.Series).stack() for x in df.columns]
df_stack = pd.concat(df_stack,1).reset_index(level=1,drop=True)
df_stack.columns=df.columns

In [1141]:
df_pred = df_stack[['date', 'pred_1', 'pred_2']]
df_pred.head()

,date,pred_1,pred_2
0,2023-06-22T01:00:00+09:00,0.0,0.0
0,2023-06-22T02:00:00+09:00,0.0,0.0
0,2023-06-22T03:00:00+09:00,0.0,0.0
0,2023-06-22T04:00:00+09:00,0.0,0.0
0,2023-06-22T05:00:00+09:00,0.0,0.0


In [1142]:
s_info = df_stack.dropna().drop(columns = ['date'])
s_info = s_info[['tilt', 'azimuth', 'plant_id', 'plant_name', 'capacity', 'ref_h']]

In [1143]:
pred_data = pd.merge(df_pred, s_info, left_index=True, right_index=True, how='left')
pred_data.reset_index(drop=True, inplace=True)

In [1144]:
pred_data['date'] = pred_data['date'].apply(lambda x: pd.to_datetime(x).tz_localize('UTC').tz_convert('Asia/Seoul'))

In [1145]:
'''
custom
'''

final_pred_data = pred_data.copy()
final_pred_data['plant_id'] = 'SMRE'
final_pred_data = final_pred_data.rename(columns={'date':'dt'})

In [1146]:
display(final_pred_data.head(3))
display(final_pred_data.tail(3))

,dt,pred_1,pred_2,tilt,azimuth,plant_id,plant_name,capacity,ref_h
0,2023-06-22 01:00:00+09:00,0.0,0.0,15.0,180.0,SMRE,신안 임자태양광발전소,99986.25,12.0
1,2023-06-22 02:00:00+09:00,0.0,0.0,15.0,180.0,SMRE,신안 임자태양광발전소,99986.25,12.0
2,2023-06-22 03:00:00+09:00,0.0,0.0,15.0,180.0,SMRE,신안 임자태양광발전소,99986.25,12.0


,dt,pred_1,pred_2,tilt,azimuth,plant_id,plant_name,capacity,ref_h
21,2023-06-22 22:00:00+09:00,0.0,0.0,15.0,180.0,SMRE,신안 임자태양광발전소,99986.25,12.0
22,2023-06-22 23:00:00+09:00,0.0,0.0,15.0,180.0,SMRE,신안 임자태양광발전소,99986.25,12.0
23,2023-06-23 00:00:00+09:00,0.0,0.0,15.0,180.0,SMRE,신안 임자태양광발전소,99986.25,12.0


# STEP 2: OPENWEATHER API 예측값

In [1147]:
dt_index = pd.date_range(start=target_start_dt,
                         end=target_end_dt,
                         freq = '1D')
dt_index

DatetimeIndex(['2023-06-22 00:00:00+09:00'], dtype='datetime64[ns, UTC+09:00]', freq='D')

In [1148]:
base_path = '/mnt/loki/data/mjhwang/project/loader/api_load/data/openweathermap/dayahead/09/'

irr_data_1 = pd.DataFrame()
wea_data_1 = pd.DataFrame()
both_data_1 = pd.DataFrame()

irr_data_2 = pd.DataFrame()
wea_data_2 = pd.DataFrame()
both_data_2 = pd.DataFrame()

# 1차 예측 (오전 9시)
for dt in dt_index:
    _dt = dt.strftime('%y%m%d')
    if (dt.day <9)&(dt.month==6):
        try:
            # 둘중 하나라도 없으면 안됨
            _irr_data = pd.read_csv(base_path+'irr_hourly_10602_{}_0900.csv'.format(_dt))
            irr_data_1 = irr_data_1.append(_irr_data)
            _wea_data = pd.read_csv(base_path+'wea_hourly_10602_{}_0900.csv'.format(_dt))
            wea_data_1 = wea_data_1.append(_wea_data)
        except:
            pass
        
    else:
        try:
            _both_data = pd.read_csv(base_path+'base_weather_hourly_10602_{}_0900.csv'.format(_dt))
            both_data_1 = both_data_1.append(_both_data)
        except:
            pass
        
        
# 2차 예측 (오전 9시)
base_path = '/mnt/loki/data/mjhwang/project/loader/api_load/data/openweathermap/dayahead/16/'
for dt in dt_index:
    _dt = dt.strftime('%y%m%d')
    if (dt.day <8) & (dt.month==6):
        try:
            # 둘중 하나라도 없으면 안됨
            _irr_data = pd.read_csv(base_path+'irr_hourly_10602_{}_1600.csv'.format(_dt))
            irr_data_2 = irr_data_2.append(_irr_data)
            _wea_data = pd.read_csv(base_path+'wea_hourly_10602_{}_1600.csv'.format(_dt))
            wea_data_2 = wea_data_2.append(_wea_data)
        except:
            pass
        
    else:
        try:
            _both_data = pd.read_csv(base_path+'base_weather_hourly_10602_{}_1600.csv'.format(_dt))
            both_data_2 = both_data_2.append(_both_data)
        except:
            pass

# irr_data 데이터의 dt를 kst로 바꿔준다.
# irr_data와 wea_data는 6월만 해당
# irr_data_1['dt'] = irr_data_1['dt'].apply(lambda x: pd.Timestamp(x, tz='Asia/Seoul'))
# irr_data_2['dt'] = irr_data_2['dt'].apply(lambda x: pd.Timestamp(x, tz='Asia/Seoul'))
# wea_data_1['dt_KST'] = wea_data_1['dt_KST'].apply(lambda x: pd.Timestamp(x, tz='Asia/Seoul'))
# wea_data_2['dt_KST'] = wea_data_2['dt_KST'].apply(lambda x: pd.Timestamp(x, tz='Asia/Seoul'))
both_data_1['dt'] = both_data_1['dt'].apply(lambda x: pd.Timestamp(x, tz='Asia/Seoul'))
both_data_2['dt'] = both_data_2['dt'].apply(lambda x: pd.Timestamp(x, tz='Asia/Seoul'))

In [1149]:
# 1차 예측 합치기
# both_data1은 6월만
# both_data1 = irr_data_1[['dt','lat','lon','cloudy_ghi','cloudy_dni','cloudy_dhi']].merge(wea_data_1[['dt_KST','temp_air','wind_speed']],
#                                                                                          left_on='dt',
#                                                                                          right_on='dt_KST',
#                                                                                          how='outer')

# both_data1 = both_data1[['dt','lat','lon','cloudy_ghi','cloudy_dni','cloudy_dhi','temp_air','wind_speed']]
both_data2 = both_data_1[['dt','lat','lon','cloudy_ghi','cloudy_dni','cloudy_dhi','temp_air','wind_speed']]
# total_data_1 = both_data1.append(both_data2)
total_data_1 = both_data2.copy()
total_data_1['chasu']=1

In [1150]:
# 2차 예측 합치기
# both_data1은 6월만
# both_data1 = irr_data_2[['dt','lat','lon','cloudy_ghi','cloudy_dni','cloudy_dhi']].merge(wea_data_2[['dt_KST','temp_air','wind_speed']],
#                                                                                          left_on='dt',
#                                                                                          right_on='dt_KST',
#                                                                                          how='outer')

# both_data1 = both_data1[['dt','lat','lon','cloudy_ghi','cloudy_dni','cloudy_dhi','temp_air','wind_speed']]
both_data2 = both_data_2[['dt','lat','lon','cloudy_ghi','cloudy_dni','cloudy_dhi','temp_air','wind_speed']]
# total_data_2 = both_data1.append(both_data2)
total_data_2 = both_data2.copy()
total_data_2['chasu']=2

In [1151]:
total_data = total_data_1.append(total_data_2)

In [1152]:
display(total_data.shape)
display(total_data.head(3))
display(total_data.tail(3))

(48, 9)

,dt,lat,lon,cloudy_ghi,cloudy_dni,cloudy_dhi,temp_air,wind_speed,chasu
0,2023-06-22 00:00:00+09:00,35.099234,126.11834,0.0,0.0,0.0,20.14,4.13,1
1,2023-06-22 01:00:00+09:00,35.099234,126.11834,0.0,0.0,0.0,20.15,4.33,1
2,2023-06-22 02:00:00+09:00,35.099234,126.11834,0.0,0.0,0.0,20.14,4.53,1


,dt,lat,lon,cloudy_ghi,cloudy_dni,cloudy_dhi,temp_air,wind_speed,chasu
21,2023-06-22 21:00:00+09:00,35.099234,126.11834,0.0,0.0,0.0,21.15,2.12,2
22,2023-06-22 22:00:00+09:00,35.099234,126.11834,0.0,0.0,0.0,21.01,2.11,2
23,2023-06-22 23:00:00+09:00,35.099234,126.11834,0.0,0.0,0.0,20.77,1.38,2


In [1153]:
total_data.sort_values(by='dt').tail(3)

,dt,lat,lon,cloudy_ghi,cloudy_dni,cloudy_dhi,temp_air,wind_speed,chasu
22,2023-06-22 22:00:00+09:00,35.099234,126.11834,0.0,0.0,0.0,20.82,3.86,1
23,2023-06-22 23:00:00+09:00,35.099234,126.11834,0.0,0.0,0.0,20.72,3.12,1
23,2023-06-22 23:00:00+09:00,35.099234,126.11834,0.0,0.0,0.0,20.77,1.38,2


In [1154]:
total_data.shape

(48, 9)

In [1155]:
start = total_data['dt'].min().strftime('%Y%m%d')
end = total_data['dt'].max().strftime('%Y%m%d')
# total_data.to_csv('data/openweather_{}_{}.csv'.format(start, end),index=False)

In [1156]:
# 발전량으로 바꾸기
data = total_data.copy()
data['dt'] = data['dt'].apply(lambda x: pd.Timestamp(x, tz='Asia/Seoul'))
data = data.set_index('dt')
data = data.rename(columns={'cloudy_ghi':'ghi',
                            'cloudy_dni':'dni',
                            'cloudy_dhi':'dhi'})

In [1157]:
display(data.shape)
display(data.head())
display(data.tail())

(48, 8)

,lat,lon,ghi,dni,dhi,temp_air,wind_speed,chasu
dt,,,,,,,,
2023-06-22 00:00:00+09:00,35.099234,126.11834,0.0,0.0,0.0,20.14,4.13,1
2023-06-22 01:00:00+09:00,35.099234,126.11834,0.0,0.0,0.0,20.15,4.33,1
2023-06-22 02:00:00+09:00,35.099234,126.11834,0.0,0.0,0.0,20.14,4.53,1
2023-06-22 03:00:00+09:00,35.099234,126.11834,0.0,0.0,0.0,20.06,5.29,1
2023-06-22 04:00:00+09:00,35.099234,126.11834,0.0,0.0,0.0,20.04,4.96,1


,lat,lon,ghi,dni,dhi,temp_air,wind_speed,chasu
dt,,,,,,,,
2023-06-22 19:00:00+09:00,35.099234,126.11834,199.48,518.19,67.75,21.57,2.86,2
2023-06-22 20:00:00+09:00,35.099234,126.11834,28.54,91.01,22.88,21.37,2.20,2
2023-06-22 21:00:00+09:00,35.099234,126.11834,0.00,0.00,0.00,21.15,2.12,2
2023-06-22 22:00:00+09:00,35.099234,126.11834,0.00,0.00,0.00,21.01,2.11,2
2023-06-22 23:00:00+09:00,35.099234,126.11834,0.00,0.00,0.00,20.77,1.38,2


In [1158]:
data_01 = data.loc[data['chasu']==1]
data_02 = data.loc[data['chasu']==2]
display(data_01.shape)
display(data_02.shape)

(24, 8)

(24, 8)

In [1159]:
# 발전소 정보
lat, lng = data[['lat','lon']].iloc[0]
cap = 99986.25
tilt = 15
azimuth = 180
lat, lng, cap, tilt, azimuth

(35.099234, 126.11833999999999, 99986.25, 15, 180)

In [1160]:
# 1차
preproc_model_01 = PreprocessData(data_01[['temp_air', 'wind_speed']],
                                  lat, lng, timezone='Asia/Seoul')
preproc_data_01 = preproc_model_01.get_processed_data(ghi=data_01['ghi'])

# 2차
preproc_model_02 = PreprocessData(data_02[['temp_air', 'wind_speed']],
                                  lat, lng, timezone='Asia/Seoul')
preproc_data_02 = preproc_model_02.get_processed_data(ghi=data_02['ghi'])

In [1161]:
display(preproc_data_01.shape)
display(preproc_data_02.shape)
display(preproc_data_01.head())
display(preproc_data_02.head())

(24, 5)

(24, 5)

,temp_air,wind_speed,ghi,dni,dhi
dt,,,,,
2023-06-22 00:00:00+09:00,20.14,4.13,0.0,0.0,0.0
2023-06-22 01:00:00+09:00,20.15,4.33,0.0,0.0,0.0
2023-06-22 02:00:00+09:00,20.14,4.53,0.0,0.0,0.0
2023-06-22 03:00:00+09:00,20.06,5.29,0.0,0.0,0.0
2023-06-22 04:00:00+09:00,20.04,4.96,0.0,0.0,0.0


,temp_air,wind_speed,ghi,dni,dhi
dt,,,,,
2023-06-22 00:00:00+09:00,19.94,2.40,0.0,0.0,0.0
2023-06-22 01:00:00+09:00,19.98,3.56,0.0,0.0,0.0
2023-06-22 02:00:00+09:00,20.00,4.05,0.0,0.0,0.0
2023-06-22 03:00:00+09:00,20.06,4.15,0.0,0.0,0.0
2023-06-22 04:00:00+09:00,20.11,4.05,0.0,0.0,0.0


In [1162]:
pvs = PVSkeleton(
    lat, lng,
    capacity=cap*1000, surface_tilt=tilt, surface_azimuth=azimuth,
    losses_model='no_loss', eta_inv_nom=0.98, altitude=0.1, timezone='UTC', albedo=0.35,
    pdc0=260,
)

In [1163]:
pv_yield_01 = pvs(preproc_data_01).to_frame('pred_yield')
pv_yield_01['pred_yield'] *= 0.9132143619059999
pv_yield_01['capacity'] = cap
pv_yield_01['chasu']=1
pv_yield_01['ghi'] = data_01['ghi']
pv_yield_01['pred_yield'] = pv_yield_01['pred_yield'].astype('float64')

pv_yield_02 = pvs(preproc_data_02).to_frame('pred_yield')
pv_yield_02['pred_yield'] *= 0.9132143619059999
pv_yield_02['capacity'] = cap
pv_yield_02['chasu']=2
pv_yield_02['ghi'] = data_02['ghi']
pv_yield_02['pred_yield'] = pv_yield_02['pred_yield'].astype('float64')

In [1164]:
display(pv_yield_01.shape)
display(data_01.shape)
display(pv_yield_02.shape)
display(data_02.shape)

(24, 4)

(24, 8)

(24, 4)

(24, 8)

In [1165]:
pv_yield = pv_yield_01.append(pv_yield_02)

In [1166]:
pv_yield1 = pv_yield.loc[pv_yield['chasu']==1]
pv_yield1 = pv_yield1.rename(columns={'pred_yield':'op_pred_1',
                                      'ghi':'op_ghi_1'}).reset_index()
pv_yield2 = pv_yield.loc[pv_yield['chasu']==2]
pv_yield2 = pv_yield2.rename(columns={'pred_yield':'op_pred_2',
                                      'ghi':'op_ghi_2'}).reset_index()

In [1167]:
pv_yield = pv_yield1[['dt','op_pred_1','op_ghi_1']].merge(pv_yield2[['dt','op_pred_2','op_ghi_2']],
                                                    on='dt',
                                                    how='outer')

In [1168]:
print(pv_yield.shape)
pv_yield.head(3)

(24, 5)


,dt,op_pred_1,op_ghi_1,op_pred_2,op_ghi_2
0,2023-06-22 00:00:00+09:00,0.0,0.0,0.0,0.0
1,2023-06-22 01:00:00+09:00,0.0,0.0,0.0,0.0
2,2023-06-22 02:00:00+09:00,0.0,0.0,0.0,0.0


In [1169]:
pv_yield.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 24 entries, 0 to 23
Data columns (total 5 columns):
dt           24 non-null datetime64[ns, Asia/Seoul]
op_pred_1    24 non-null float64
op_ghi_1     24 non-null float64
op_pred_2    24 non-null float64
op_ghi_2     24 non-null float64
dtypes: datetime64[ns, Asia/Seoul](1), float64(4)
memory usage: 1.1 KB


# STEP3: SMRE TRUE GEN DATA

In [1170]:
SMRE_token = 'lqyolpzUdyM2ocqOLcb4CAxme10'
url = "https://ensolgrid.skens.com/rest/v1/report/data/stat"

In [1171]:
def make_datetime(start, end):
    start = pd.Timestamp(start).tz_convert(tz='UTC')-timedelta(hours=1)
    end = pd.Timestamp(end).tz_convert(tz='UTC')-timedelta(hours=1)+timedelta(days=1)
    
    # unix 시간으로 변환해주기
    start_time = int(time.mktime(start.timetuple())*1000)
    end_time = int(time.mktime(end.timetuple())*1000)
    return start_time, end_time

In [1172]:
def make_smre_gen(start,end):
    
    start_time = str(start)
    end_time = str(end)
    print(start_time)
    print(end_time)

    header = {
    "Authorization":"bearer "+SMRE_token}

    body = {
    "keyType":"sn",
    "keySuffix":"_METER",
    "rIds":"AB1K",
    "reportRIds":"ACTIVE_ENERGY",
    "startDate": start_time,
    "endDate":end_time,   
    "timeIntervalUnit":"RAW"}
    
    res = requests.get(url, headers = header, params=body)
    print(res)
#     while res.status_code != 200:
#         time.sleep(2) # 2초 후 다시 시도
#         res = requests.get(url, headers = header, params=body)
    data = res.json()
    data = json_normalize(data['AB1K'],
                          record_path=['ACTIVE_ENERGY'])
    
    data['dt'] = pd.to_datetime(data['timestamp'],unit='ms')+timedelta(hours=1)
    data['dt'] = pd.DatetimeIndex(data['dt'],tz='UTC').tz_convert('Asia/Seoul')
    return data

In [1173]:
def main():
    # API 정보
    SMRE_token = 'lqyolpzUdyM2ocqOLcb4CAxme10'
    url = "https://ensolgrid.skens.com/rest/v1/report/data/stat"
    
    # start, end 설정
    # datetime.now()는 UTC, 계산하기 쉽게 KST로 바꿔줌
    date1 = pd.Timestamp(datetime.now()).tz_localize(tz='UTC').tz_convert('Asia/Seoul')-timedelta(days=1)
    date2 = pd.Timestamp(datetime.now()).tz_localize(tz='UTC').tz_convert('Asia/Seoul')

    start = datetime(date1.year, date1.month, date1.day, 1)
    end = datetime(date2.year, date2.month, date2.day)
    
    # 날짜 UNIX 형태로 바꿔주기
    start_time, end_time = make_datetime(start,end)
    
    # 해당하는 날짜에 대한 발전량값 구하기
    data = make_smre_gen(start_time, end_time)

    return data

In [1174]:
# 시간 바꾸기
start = target_start_dt
end = target_end_dt
start_time, end_time = make_datetime(start,end)
start_time, end_time 

(1687356000000, 1687442400000)

In [1175]:
display(pd.to_datetime(start_time,unit='ms')+timedelta(hours=1))
pd.to_datetime(end_time,unit='ms')+timedelta(hours=1)

Timestamp('2023-06-21 15:00:00')

Timestamp('2023-06-22 15:00:00')

In [1176]:
# 해당하는 날짜에 대한 발전량값 구하기
smre_data = make_smre_gen(start_time, end_time)

1687356000000
1687442400000
<Response [200]>


In [1177]:
display(smre_data.head(3))
display(smre_data.tail(3))

,avg,count,max,min,sum,timestamp,dt
0,0.0,1,0.0,0.0,0.0,1687356000000,2023-06-22 00:00:00+09:00
1,0.0,1,0.0,0.0,0.0,1687359600000,2023-06-22 01:00:00+09:00
2,0.0,1,0.0,0.0,0.0,1687363200000,2023-06-22 02:00:00+09:00


,avg,count,max,min,sum,timestamp,dt
22,0.0,1,0.0,0.0,0.0,1687435200000,2023-06-22 22:00:00+09:00
23,0.0,1,0.0,0.0,0.0,1687438800000,2023-06-22 23:00:00+09:00
24,0.0,1,0.0,0.0,0.0,1687442400000,2023-06-23 00:00:00+09:00


# STEP 4: METEOLOGICA 데이터

In [1178]:
target_start_dt2 = target_start_dt-timedelta(days=1)
target_end_dt2 = target_end_dt-timedelta(days=1)

year = target_start_dt2.year
month = target_start_dt2.month
day = target_start_dt2.day
start_date = '{0}{1:02d}{2:02d}'.format(year,month,day)

year = target_end_dt2.year
month = target_end_dt2.month
day = target_end_dt2.day
end_date = '{0}{1:02d}{2:02d}'.format(year,month,day)

In [1179]:
dt_index = pd.date_range(start=start_date,
                         end=end_date,
                         freq='1D')
dt_index

DatetimeIndex(['2023-06-21'], dtype='datetime64[ns]', freq='D')

In [1180]:
start_date

'20230621'

In [1181]:
# 예측 날짜 설정부
# start_date = '{}{:02d}03'.format(year, 7)
# end_date = '{}{:02d}03'.format(year, 7)
dt_index = pd.date_range(start=start_date,
                         end=end_date,
                         freq='1D')
display(dt_index)

import datetime
# 유닉스 시간변환
def unixtime_to_localtime(unix_time):
    # UNIX 시간을 datetime 객체로 변환
    utc_datetime = datetime.datetime.utcfromtimestamp(unix_time)

    # UTC 시간을 로컬 시간으로 변환
    local_datetime = utc_datetime.replace(tzinfo=datetime.timezone.utc).astimezone(tz=None)

    return local_datetime

DatetimeIndex(['2023-06-21'], dtype='datetime64[ns]', freq='D')

In [1182]:
# 1차
total_1cha = pd.DataFrame()

for date_i in dt_index:
    print(date_i)
    timezone = pytz.timezone("Asia/Seoul")
    username = "haezoom_api"
    password = "TXw72j{4"

    client = Client("https://webservice.meteologica.com/api/wsdl/MeteologicaDataExchangeService.wsdl")

    login_params = {"username": username, "password": password}

    # request login
    login_response = client.service.login(login_params)

    # save session token
    session_token = login_response.header.sessionToken

    # check login error
    error_code = login_response.errorCode
    if error_code != "OK":
        print(f"Login failed: {error_code}")
    else:
        print("Login success")

    # wind power id
#     facility_id = "haezoom_wind"
    facility_id = "pv10"
    variable_id = "prod"
    predictor_id = "aggregated"

    params = {"header": {"sessionToken": session_token}, "facilityId": facility_id}

    result = client.service.getFacilityForecastedVnP(params)

    day_forcast = pd.DataFrame()
    day_forcast['date'] = []
    day_forcast['pred'] = []
    fdate = date_i + timedelta(days = 1)
    fadate = fdate + timedelta(days = 1)
    
    from_date = date_i + timedelta(days=1)
    params = {
        "header": {"sessionToken": session_token},
        # "facilityId": facility_id,
        "variableId": variable_id,
        "predictorId": predictor_id,
        "facilitiesId": [{"item": facility_id}],

        "forecastDate": f"{date_i.year}-{date_i.month:02d}-{date_i.day:02d}T09:00:00+09:00",  # 언제 예측한 것인지 (예측 시점)
        "fromDate": f"{fdate.year}-{fdate.month:02d}-{fdate.day:02d}T00:00:00+09:00",  # 예측 시작 시점
        "toDate": f"{fadate.year}-{fadate.month:02d}-{fadate.day:02d}T00:00:00+09:00",  # 예측 종료 시점
#         "forecastDate": f"{date_i.strftime('%Y-%m-%dT%H:%M:%S+09:00')}",  # 언제 예측한 것인지 (예측 시점)        
#         "fromDate": f"{date_i + timedelta(days =1).strftime('%Y-%m-%dT%H:%M:%S+09:00')}",  # 예측 시작 시점
#         "toDate": f"{date_i + timedelta(days =2).strftime('%Y-%m-%dT%H:%M:%S+09:00')}",  # 예측 시작 시점
        # granularity='hourly',
        "percentiles": "50",  # 값은 중앙값만 사용한다 (perc50)
    }

    result = client.service.getForecastMulti(params)
    data = result["facilitiesForecastData"]["item"][0]["forecastData"]
    index = 0
    temp_data = pd.DataFrame()
    for row in data.split(":"):
        vars = row.split("~")
        if len(vars) > 1:
            date, *values = vars

#             day_forcast[unixtime_to_localtime(int(date))] = values
#             day_forcast['date'] = unixtime_to_localtime(int(date))
#             day_forcast['pred'] = values
#             print(unixtime_to_localtime(int(date)), values)
#             print(day_forcast)
            temp_data['date'] = pd.to_datetime(unixtime_to_localtime(int(date))) # date부분에서 에러가 나면 들여쓰기해서 실행했다가 다시 빼주기
            temp_data['pred'] = values
        day_forcast = day_forcast.append(temp_data)
#         print(day_forcast)
#     display(day_forcast)
    total_1cha = total_1cha.append(day_forcast)
# total_1cha

total_1cha = total_1cha.reset_index(drop = True)

test_data_1 = total_1cha.drop(total_1cha.index[0])

test_data_1.date =test_data_1.date + timedelta(hours = 1)
test_data_1 = test_data_1.iloc[:-1,:]
test_data_1 = test_data_1.rename(columns = {'date':'Date_time'})
test_data_1['Date_time'] = test_data_1['Date_time'].apply(lambda x: pd.Timestamp(x, tz='Asia/Seoul'))
test_data_1

2023-06-21 00:00:00
Login success


,Date_time,pred
1,2023-06-22 02:00:00+09:00,0
2,2023-06-22 03:00:00+09:00,0
3,2023-06-22 04:00:00+09:00,0
4,2023-06-22 05:00:00+09:00,0
5,2023-06-22 06:00:00+09:00,559
6,2023-06-22 07:00:00+09:00,5743
7,2023-06-22 08:00:00+09:00,15810
8,2023-06-22 09:00:00+09:00,28887
9,2023-06-22 10:00:00+09:00,43019
10,2023-06-22 11:00:00+09:00,54636


In [1183]:
total_1cha = total_1cha.reset_index(drop = True)

test_data_1 = total_1cha.drop(total_1cha.index[0])

test_data_1.date =test_data_1.date + timedelta(hours = 1)
test_data_1 = test_data_1.iloc[:-1,:]
test_data_1 = test_data_1.rename(columns = {'date':'Date_time'})
test_data_1['Date_time'] = test_data_1['Date_time'].apply(lambda x: pd.Timestamp(x, tz='Asia/Seoul'))
test_data_1

,Date_time,pred
1,2023-06-22 02:00:00+09:00,0
2,2023-06-22 03:00:00+09:00,0
3,2023-06-22 04:00:00+09:00,0
4,2023-06-22 05:00:00+09:00,0
5,2023-06-22 06:00:00+09:00,559
6,2023-06-22 07:00:00+09:00,5743
7,2023-06-22 08:00:00+09:00,15810
8,2023-06-22 09:00:00+09:00,28887
9,2023-06-22 10:00:00+09:00,43019
10,2023-06-22 11:00:00+09:00,54636


In [1184]:
# 2차
total_2cha = pd.DataFrame()

for date_i in dt_index:
    print(date_i)
    timezone = pytz.timezone("Asia/Seoul")
    username = "haezoom_api"
    password = "TXw72j{4"

    client = Client("https://webservice.meteologica.com/api/wsdl/MeteologicaDataExchangeService.wsdl")

    login_params = {"username": username, "password": password}

    # request login
    login_response = client.service.login(login_params)

    # save session token
    session_token = login_response.header.sessionToken

    # check login error
    error_code = login_response.errorCode
    if error_code != "OK":
        print(f"Login failed: {error_code}")
    else:
        print("Login success")

    # wind power id
#     facility_id = "haezoom_wind"
    facility_id = "pv10"
    variable_id = "prod"
    predictor_id = "aggregated"

    params = {"header": {"sessionToken": session_token}, "facilityId": facility_id}

    result = client.service.getFacilityForecastedVnP(params)

    day_forcast = pd.DataFrame()
    day_forcast['date'] = []
    day_forcast['pred'] = []
    fdate = date_i + timedelta(days = 1)
    fadate = fdate + timedelta(days = 1)
    
    from_date = date_i + timedelta(days=1)
    params = {
        "header": {"sessionToken": session_token},
        # "facilityId": facility_id,
        "variableId": variable_id,
        "predictorId": predictor_id,
        "facilitiesId": [{"item": facility_id}],

        "forecastDate": f"{date_i.year}-{date_i.month:02d}-{date_i.day:02d}T16:00:00+09:00",  # 언제 예측한 것인지 (예측 시점)
        "fromDate": f"{fdate.year}-{fdate.month:02d}-{fdate.day:02d}T00:00:00+09:00",  # 예측 시작 시점
        "toDate": f"{fadate.year}-{fadate.month:02d}-{fadate.day:02d}T00:00:00+09:00",  # 예측 종료 시점
#         "forecastDate": f"{date_i.strftime('%Y-%m-%dT%H:%M:%S+09:00')}",  # 언제 예측한 것인지 (예측 시점)        
#         "fromDate": f"{date_i + timedelta(days =1).strftime('%Y-%m-%dT%H:%M:%S+09:00')}",  # 예측 시작 시점
#         "toDate": f"{date_i + timedelta(days =2).strftime('%Y-%m-%dT%H:%M:%S+09:00')}",  # 예측 시작 시점
        # granularity='hourly',
        "percentiles": "50",  # 값은 중앙값만 사용한다 (perc50)
    }

    result = client.service.getForecastMulti(params)

    data = result["facilitiesForecastData"]["item"][0]["forecastData"]
    index = 0
    temp_data = pd.DataFrame()
    for row in data.split(":"):

        vars = row.split("~")
        if len(vars) > 1:
            date, *values = vars
#             day_forcast[unixtime_to_localtime(int(date))] = values
#             day_forcast['date'] = unixtime_to_localtime(int(date))
#             day_forcast['pred'] = values
#             print(unixtime_to_localtime(int(date)), values)
#             print(day_forcast)
        temp_data['date'] = pd.to_datetime(unixtime_to_localtime(int(date)))
        temp_data['pred'] = values
        day_forcast = day_forcast.append(temp_data)
#         print(day_forcast)
#     display(day_forcast)
    total_2cha = total_2cha.append(day_forcast)
total_2cha

total_2cha = total_2cha.reset_index(drop = True)

test_data_2 = total_2cha.drop(total_2cha.index[0])

test_data_2.date =test_data_2.date + timedelta(hours = 1)
test_data_2 = test_data_2.iloc[:-1,:]
test_data_2 = test_data_2.rename(columns = {'date':'Date_time'})
test_data_2['Date_time'] = test_data_2['Date_time'].apply(lambda x: pd.Timestamp(x, tz='Asia/Seoul'))
test_data_2

2023-06-21 00:00:00
Login success


,Date_time,pred
1,2023-06-22 01:00:00+09:00,0
2,2023-06-22 02:00:00+09:00,0
3,2023-06-22 03:00:00+09:00,0
4,2023-06-22 04:00:00+09:00,0
5,2023-06-22 05:00:00+09:00,0
6,2023-06-22 06:00:00+09:00,601
7,2023-06-22 07:00:00+09:00,5962
8,2023-06-22 08:00:00+09:00,15868
9,2023-06-22 09:00:00+09:00,28904
10,2023-06-22 10:00:00+09:00,43534


In [1185]:
test_data_1 = test_data_1.loc[test_data_1['Date_time'].notnull()]
test_data_2 = test_data_2.loc[test_data_2['Date_time'].notnull()]

test_data_1 = test_data_1.rename(columns={'pred':'mt_pred_1'})
test_data_2 = test_data_2.rename(columns={'pred':'mt_pred_2'})

test_data = test_data_1.merge(test_data_2,
                              on='Date_time')

In [1186]:
test_data

,Date_time,mt_pred_1,mt_pred_2
0,2023-06-22 02:00:00+09:00,0,0
1,2023-06-22 03:00:00+09:00,0,0
2,2023-06-22 04:00:00+09:00,0,0
3,2023-06-22 05:00:00+09:00,0,0
4,2023-06-22 06:00:00+09:00,559,601
5,2023-06-22 07:00:00+09:00,5743,5962
6,2023-06-22 08:00:00+09:00,15810,15868
7,2023-06-22 09:00:00+09:00,28887,28904
8,2023-06-22 10:00:00+09:00,43019,43534
9,2023-06-22 11:00:00+09:00,54636,55319


# STEP 5: 전부 합치기

In [1187]:
# HAEZOOM Day-Ahead
display(final_pred_data.shape)
display(final_pred_data.head(3))
display(final_pred_data.tail(3))

(24, 9)

,dt,pred_1,pred_2,tilt,azimuth,plant_id,plant_name,capacity,ref_h
0,2023-06-22 01:00:00+09:00,0.0,0.0,15.0,180.0,SMRE,신안 임자태양광발전소,99986.25,12.0
1,2023-06-22 02:00:00+09:00,0.0,0.0,15.0,180.0,SMRE,신안 임자태양광발전소,99986.25,12.0
2,2023-06-22 03:00:00+09:00,0.0,0.0,15.0,180.0,SMRE,신안 임자태양광발전소,99986.25,12.0


,dt,pred_1,pred_2,tilt,azimuth,plant_id,plant_name,capacity,ref_h
21,2023-06-22 22:00:00+09:00,0.0,0.0,15.0,180.0,SMRE,신안 임자태양광발전소,99986.25,12.0
22,2023-06-22 23:00:00+09:00,0.0,0.0,15.0,180.0,SMRE,신안 임자태양광발전소,99986.25,12.0
23,2023-06-23 00:00:00+09:00,0.0,0.0,15.0,180.0,SMRE,신안 임자태양광발전소,99986.25,12.0


In [1188]:
# OPENWEATHER Day-Ahead
display(pv_yield.shape)
display(pv_yield.head(3))
display(pv_yield.tail(3))

(24, 5)

,dt,op_pred_1,op_ghi_1,op_pred_2,op_ghi_2
0,2023-06-22 00:00:00+09:00,0.0,0.0,0.0,0.0
1,2023-06-22 01:00:00+09:00,0.0,0.0,0.0,0.0
2,2023-06-22 02:00:00+09:00,0.0,0.0,0.0,0.0


,dt,op_pred_1,op_ghi_1,op_pred_2,op_ghi_2
21,2023-06-22 21:00:00+09:00,753.506154,0.0,1029.752514,0.0
22,2023-06-22 22:00:00+09:00,0.000000,0.0,0.000000,0.0
23,2023-06-22 23:00:00+09:00,0.000000,0.0,0.000000,0.0


In [1189]:
# SMRE TRUE Day-Ahead
display(smre_data.shape)
display(smre_data.head(3))
display(smre_data.tail(3))

(25, 7)

,avg,count,max,min,sum,timestamp,dt
0,0.0,1,0.0,0.0,0.0,1687356000000,2023-06-22 00:00:00+09:00
1,0.0,1,0.0,0.0,0.0,1687359600000,2023-06-22 01:00:00+09:00
2,0.0,1,0.0,0.0,0.0,1687363200000,2023-06-22 02:00:00+09:00


,avg,count,max,min,sum,timestamp,dt
22,0.0,1,0.0,0.0,0.0,1687435200000,2023-06-22 22:00:00+09:00
23,0.0,1,0.0,0.0,0.0,1687438800000,2023-06-22 23:00:00+09:00
24,0.0,1,0.0,0.0,0.0,1687442400000,2023-06-23 00:00:00+09:00


In [1190]:
# METEOLOGICAL TRUE Day-Ahead
display(test_data.shape)
display(test_data.head(3))
display(test_data.tail(3))

(23, 3)

,Date_time,mt_pred_1,mt_pred_2
0,2023-06-22 02:00:00+09:00,0,0
1,2023-06-22 03:00:00+09:00,0,0
2,2023-06-22 04:00:00+09:00,0,0


,Date_time,mt_pred_1,mt_pred_2
20,2023-06-22 22:00:00+09:00,0,0
21,2023-06-22 23:00:00+09:00,0,0
22,2023-06-23 00:00:00+09:00,0,0


In [1191]:
total_data = final_pred_data.merge(pv_yield,
                                   on='dt',
                                   how='outer')
total_data = total_data.merge(smre_data[['dt','sum']],
                              on='dt',
                              how='outer')
total_data = total_data.merge(test_data,
                              left_on='dt',
                              right_on='Date_time',
                              how='outer')

In [1192]:
display(total_data.shape)
display(total_data.head(3))
display(total_data.tail(3))

(25, 17)

,dt,pred_1,pred_2,tilt,azimuth,plant_id,plant_name,capacity,ref_h,op_pred_1,op_ghi_1,op_pred_2,op_ghi_2,sum,Date_time,mt_pred_1,mt_pred_2
0,2023-06-22 01:00:00+09:00,0.0,0.0,15.0,180.0,SMRE,신안 임자태양광발전소,99986.25,12.0,0.0,0.0,0.0,0.0,0.0,NaT,NaN,NaN
1,2023-06-22 02:00:00+09:00,0.0,0.0,15.0,180.0,SMRE,신안 임자태양광발전소,99986.25,12.0,0.0,0.0,0.0,0.0,0.0,2023-06-22 02:00:00+09:00,0,0
2,2023-06-22 03:00:00+09:00,0.0,0.0,15.0,180.0,SMRE,신안 임자태양광발전소,99986.25,12.0,0.0,0.0,0.0,0.0,0.0,2023-06-22 03:00:00+09:00,0,0


,dt,pred_1,pred_2,tilt,azimuth,plant_id,plant_name,capacity,ref_h,op_pred_1,op_ghi_1,op_pred_2,op_ghi_2,sum,Date_time,mt_pred_1,mt_pred_2
22,2023-06-22 23:00:00+09:00,0.0,0.0,15.0,180.0,SMRE,신안 임자태양광발전소,99986.25,12.0,0.0,0.0,0.0,0.0,0.0,2023-06-22 23:00:00+09:00,0,0
23,2023-06-23 00:00:00+09:00,0.0,0.0,15.0,180.0,SMRE,신안 임자태양광발전소,99986.25,12.0,NaN,NaN,NaN,NaN,0.0,2023-06-23 00:00:00+09:00,0,0
24,2023-06-22 00:00:00+09:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaT,NaN,NaN


In [1193]:
day = start.strftime('%Y%m%d')
display(day)
total_data.to_csv('data/TOTAL_SMRE/SMRE_{}.csv'.format(day),index=False)

'20230622'